Facebook AI Similarity Search (Faiss) is a library for efficient similarity serach and clustering of dense vector. IT contains algorithm that search in sets of vectors of any size , up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter turing.

In [18]:
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'e:\GenerativeAi_python\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [19]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader('Speech_3_Technology.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=400 , chunk_overlap=30)
docs = text_splitter.split_documents(documents)
docs

[Document(metadata={'source': 'Speech_3_Technology.txt'}, page_content='The Role of Technology in Modern Society\n\nGood evening everyone. Today I would like to talk about the role of technology in modern society. Technology has become an integral part of our daily lives and continues to influence the way we learn, work, communicate, and solve problems.'),
 Document(metadata={'source': 'Speech_3_Technology.txt'}, page_content='From smartphones and computers to artificial intelligence and cloud computing, technology has transformed nearly every industry. It has made communication faster, information more accessible, and services more efficient. People can connect with others across the world within seconds and access educational resources from almost anywhere.'),
 Document(metadata={'source': 'Speech_3_Technology.txt'}, page_content='In the field of education, technology has created new opportunities for learning. Online courses, digital libraries, and virtual classrooms have made educa

In [20]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
db = FAISS.from_documents(docs , embeddings)
db

In [21]:
query = "and virtual classrooms have made education more flexible and accessible. Students can learn at their own pace and gain knowledge from experts around the globe."
docs = db.similarity_search(query)
docs[0].page_content

'In the field of education, technology has created new opportunities for learning. Online courses, digital libraries, and virtual classrooms have made education more flexible and accessible. Students can learn at their own pace and gain knowledge from experts around the globe.'

AS a Retriever
We can also convert the vercorstore into a Retriever class. This allows us to easily use it in other Langchain mathonds , which largely work with retrievers.


In [17]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'In the field of education, technology has created new opportunities for learning. Online courses, digital libraries, and virtual classrooms have made education more flexible and accessible. Students can learn at their own pace and gain knowledge from experts around the globe.'

### Similarity Search with score
There are some FAISS sepecific methods. One of them in similarity_search_with_score , which alllows you to return not only the documents but also the distance score of the query to them , The returned distance score is L2 distances. Therefore , a lower score is better.


In [22]:
docs_and_scores = db.similarity_search_with_score(query)
docs_and_scores

[(Document(id='5886fb7b-ddec-4807-be78-e335f47c1f3e', metadata={'source': 'Speech_3_Technology.txt'}, page_content='In the field of education, technology has created new opportunities for learning. Online courses, digital libraries, and virtual classrooms have made education more flexible and accessible. Students can learn at their own pace and gain knowledge from experts around the globe.'),
  np.float32(0.17677748)),
 (Document(id='e65ed71c-bdf1-4c56-ac48-8b5b9e26a720', metadata={'source': 'Speech_3_Technology.txt'}, page_content='From smartphones and computers to artificial intelligence and cloud computing, technology has transformed nearly every industry. It has made communication faster, information more accessible, and services more efficient. People can connect with others across the world within seconds and access educational resources from almost anywhere.'),
  np.float32(0.5703561)),
 (Document(id='00a8784f-f516-4291-9053-2577854df2fa', metadata={'source': 'Speech_3_Technolog

In [23]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.021953568,
 0.09583159,
 -0.18729718,
 -0.023564525,
 0.064366885,
 0.05358755,
 0.029100321,
 0.019347599,
 0.050766803,
 -0.011055268,
 0.04063241,
 0.013984981,
 0.0591378,
 0.06878115,
 0.0065289787,
 -0.04134707,
 -0.03269553,
 -0.044713106,
 -0.08222858,
 0.008790877,
 -0.052854475,
 -0.05152761,
 0.00068823894,
 -0.0046821283,
 -0.003281612,
 0.015330652,
 -0.07744661,
 -0.0025184709,
 0.0018365161,
 0.04067903,
 0.053983256,
 -0.059830643,
 0.051689196,
 0.009044877,
 -0.016108504,
 -0.007741435,
 -0.008661812,
 -0.016127624,
 -0.009948799,
 0.0051117996,
 -0.012648632,
 0.003022403,
 -0.0030987598,
 -0.019537477,
 0.06035705,
 -0.012870246,
 -0.0011565835,
 -0.037522636,
 0.06357446,
 -0.019492708,
 -0.009454878,
 -0.06086465,
 -0.018459225,
 0.033465926,
 0.0688593,
 0.020878961,
 0.0050940104,
 0.01717242,
 0.04091865,
 -0.05552686,
 0.07149029,
 0.06892979,
 -0.01774331,
 0.06491163,
 0.013108816,
 0.009387818,
 -0.06049958,
 0.053937905,
 -0.0019293579,
 -0.038582034,


In [24]:
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='5886fb7b-ddec-4807-be78-e335f47c1f3e', metadata={'source': 'Speech_3_Technology.txt'}, page_content='In the field of education, technology has created new opportunities for learning. Online courses, digital libraries, and virtual classrooms have made education more flexible and accessible. Students can learn at their own pace and gain knowledge from experts around the globe.'),
 Document(id='e65ed71c-bdf1-4c56-ac48-8b5b9e26a720', metadata={'source': 'Speech_3_Technology.txt'}, page_content='From smartphones and computers to artificial intelligence and cloud computing, technology has transformed nearly every industry. It has made communication faster, information more accessible, and services more efficient. People can connect with others across the world within seconds and access educational resources from almost anywhere.'),
 Document(id='00a8784f-f516-4291-9053-2577854df2fa', metadata={'source': 'Speech_3_Technology.txt'}, page_content='Businesses have benefited greatly

In [25]:
## Saving and Loading
db.save_local("faiss_index")

In [30]:
new_df = FAISS.load_local("faiss_index" , embeddings,allow_dangerous_deserialization=True)
docs = new_df.similarity_search(query)
docs

[Document(id='5886fb7b-ddec-4807-be78-e335f47c1f3e', metadata={'source': 'Speech_3_Technology.txt'}, page_content='In the field of education, technology has created new opportunities for learning. Online courses, digital libraries, and virtual classrooms have made education more flexible and accessible. Students can learn at their own pace and gain knowledge from experts around the globe.'),
 Document(id='e65ed71c-bdf1-4c56-ac48-8b5b9e26a720', metadata={'source': 'Speech_3_Technology.txt'}, page_content='From smartphones and computers to artificial intelligence and cloud computing, technology has transformed nearly every industry. It has made communication faster, information more accessible, and services more efficient. People can connect with others across the world within seconds and access educational resources from almost anywhere.'),
 Document(id='00a8784f-f516-4291-9053-2577854df2fa', metadata={'source': 'Speech_3_Technology.txt'}, page_content='Businesses have benefited greatly